# ADV-01: Mathematical Foundations for Advanced Computer Vision
## Singular Value Decomposition, Spectral Geometry, and Lie Groups SO(3)

**Author:** Aji Pamungkas Tri Nurcahyo (Institut Teknologi Sumatera - ITERA)  
**Level:** L3 Advanced Curriculum  
**Pillar:** Pillar 1 — Intelligent Computer Vision (ISVWS 2026–2040)  

---
### Abstract
This laboratory demonstrates the analytical foundations of advanced computer vision:
1. Matrix SVD decomposition and the Eckart-Young-Mirsky low-rank approximation theorem.
2. Condition number analysis $\kappa(A) = \sigma_{\max} / \sigma_{\min}$ and numerical stability.
3. Exponential map from Lie algebra $\mathfrak{so}(3)$ to Lie group $\text{SO}(3)$ via Rodrigues' formula.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print(f"NumPy Version: {np.__version__}")

### 1. Eckart-Young-Mirsky Low-Rank Matrix Approximation
For any matrix $A \in \mathbb{R}^{m \times n}$, the best rank-$k$ approximation under Frobenius norm is given by:
$$A_k = \sum_{i=1}^k \sigma_i u_i v_i^T, \quad \|A - A_k\|_F = \sqrt{\sum_{i=k+1}^r \sigma_i^2}$$
Let us verify this theorem empirically.

In [ ]:
np.random.seed(42)
m, n = 8, 6
A = np.random.randn(m, n) * 3.0

U, S, Vt = np.linalg.svd(A, full_matrices=False)
rank_max = len(S)

theoretical_errors = []
empirical_errors = []

for k in range(1, rank_max + 1):
    A_k = np.dot(U[:, :k], np.dot(np.diag(S[:k]), Vt[:k, :]))
    emp_err = np.linalg.norm(A - A_k, 'fro')
    theo_err = np.sqrt(np.sum(S[k:] ** 2))
    
    empirical_errors.append(emp_err)
    theoretical_errors.append(theo_err)
    print(f"Rank k={k}: Empirical Error = {emp_err:.5f}, Theoretical Bound = {theo_err:.5f}")
    assert np.isclose(emp_err, theo_err, atol=1e-5), f"Mismatch at rank {k}!"

print("\n[PASSED] Eckart-Young-Mirsky Theorem verified across all ranks!")

### 2. Spectral Energy Distribution & Condition Number
The condition number $\kappa(A) = \sigma_1 / \sigma_r$ indicates numerical sensitivity.

In [ ]:
cond_number = S[0] / S[-1]
cumulative_energy = np.cumsum(S**2) / np.sum(S**2) * 100

print(f"Singular Values: {np.round(S, 3)}")
print(f"Matrix Condition Number kappa(A): {cond_number:.2f}")
print(f"Energy captured by rank-2: {cumulative_energy[1]:.2f}%")
print(f"Energy captured by rank-3: {cumulative_energy[2]:.2f}%")

### 3. Rodrigues' Formula: Lie Algebra $\mathfrak{so}(3)$ to Lie Group $\text{SO}(3)$
Rotations in 3D are represented as Lie group elements without gimbal lock:
$$R = \exp([\boldsymbol{\omega}]_\times) = I + \frac{\sin \theta}{\theta} [\boldsymbol{\omega}]_\times + \frac{1 - \cos \theta}{\theta^2} [\boldsymbol{\omega}]_\times^2$$

In [ ]:
def rodrigues_exp(omega):
    theta = np.linalg.norm(omega)
    if theta < 1e-12:
        return np.eye(3)
    kx, ky, kz = omega / theta
    K = np.array([
        [0.0, -kz, ky],
        [kz, 0.0, -kx],
        [-ky, kx, 0.0]
    ])
    return np.eye(3) + np.sin(theta) * K + (1.0 - np.cos(theta)) * np.dot(K, K)

# 45-degree rotation around axis [1, 1, 0]
axis = np.array([1.0, 1.0, 0.0]) / np.sqrt(2.0)
angle = np.pi / 4.0
omega = axis * angle

R = rodrigues_exp(omega)
print("Rotation Matrix R:\n", np.round(R, 4))

# Verify Special Orthogonal Group properties
ortho_err = np.linalg.norm(R @ R.T - np.eye(3))
det_val = np.linalg.det(R)
print(f"\nOrthogonality Residual ||R R^T - I||: {ortho_err:.2e}")
print(f"Determinant det(R): {det_val:.6f}")

assert ortho_err < 1e-6, "Not orthogonal!"
assert np.isclose(det_val, 1.0, atol=1e-5), "Determinant is not +1!"
print("[PASSED] Valid SO(3) rotation verified!")

### 4. Summary & Research Connections
- **ADV-01** provides the analytical tooling for non-linear optimization and manifold geometry.
- These techniques directly empower **Track R01** (detection loss surfaces), **Track R02** (Kalman covariance), and **Track R06** (3D pose estimation and visual SLAM).